# AI-Powered Smart Email Management System
## Using NLP and Machine Learning

---

| Field | Details |
|-------|--------|
| **Project Title** | AI-Powered Smart Email Management System using NLP and ML |
| **Domain** | Natural Language Processing / Machine Learning |
| **Tools** | Python, Scikit-learn, HuggingFace Transformers, NLTK, Matplotlib, Plotly |
| **Platform** | Google Colab (GPU Recommended) |

---
#### Project Overview
This project builds a **production-ready Smart Email Management System** that:
- Classifies emails into 5 categories using traditional ML + BERT
- Scores email priority (0–100)
- Summarizes long emails using transformer-based NLP
- Visualizes insights via an analytics dashboard
- Provides an interactive classification interface


---
## 📦 SECTION 1: Installation & Environment Setup
---

In [ ]:
# ============================================================
# CELL 1: Install All Required Libraries
# ============================================================

print("🔧 Installing required libraries...")
print("⏳ This may take 2-3 minutes. Please wait...\n")

!pip install transformers==4.40.0 -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install datasets -q
!pip install sentencepiece -q
!pip install accelerate -q
!pip install plotly -q
!pip install wordcloud -q
!pip install scikit-learn -q
!pip install nltk -q
!pip install pandas numpy matplotlib seaborn -q
!pip install tqdm -q

print("\n✅ All libraries installed successfully!")

In [ ]:
!pip install -U transformers sentence-transformers

In [ ]:
# ============================================================
# CELL 2: Import All Libraries
# ============================================================

# ── Standard Libraries ──
import os
import re
import json
import time
import random
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
warnings.filterwarnings('ignore')

# ── NLTK ──
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer

# ── Scikit-learn ──
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score
)
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline

# ── Visualization ──
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── HuggingFace Transformers ──
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    pipeline, BertTokenizer, BertForSequenceClassification,
    DistilBertTokenizer, DistilBertForSequenceClassification,
    Trainer, TrainingArguments
)
import torch
from torch.utils.data import Dataset, DataLoader

# ── Misc ──
from tqdm import tqdm
from collections import Counter
from IPython.display import display, HTML, clear_output

# Set Seeds for Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 120)

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("="*65)
print("✅ All Libraries Imported Successfully")
print("="*65)
print(f"🖥️  Device       : {device}")
print(f"🔥 GPU Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU Name      : {torch.cuda.get_device_name(0)}")
print(f"🐍 PyTorch Ver   : {torch.__version__}")
print("="*65)

---
## 📊 SECTION 2: Dataset Creation
---

In [ ]:
# ============================================================
# CELL 3: Generate Large Realistic Email Dataset
# ============================================================

print("📧 Generating Realistic Email Dataset...\n")

# ── Template pools for each category ──

IMPORTANT_EMAILS = [
    "Dear {name}, I wanted to follow up on our meeting scheduled for {day}. The agenda includes project status, budget approval, and Q{q} targets. Please confirm your attendance by {time}.",
    "Hi {name}, This is a reminder that your {doc} document needs to be submitted before the {day} deadline. Failure to submit may affect your {outcome}. Please act immediately.",
    "Subject: URGENT - Project Deadline Tomorrow. Hi {name}, The project deliverables are due tomorrow morning. Please ensure all code is committed and the report is finalized. The manager is waiting.",
    "Hi {name}, Your interview has been scheduled for {day} at {time} with the {team} team. Please bring your resume, portfolio, and any relevant work samples. This is a final round interview.",
    "Dear {name}, Your salary payment of {amount} has been processed. Please find the payslip attached. Contact HR if you notice any discrepancies before {day}.",
    "URGENT: {name}, Your meeting with the director has been rescheduled to {day} at {time}. Attendance is mandatory. Prepare the {doc} report and key metrics.",
    "Hi {name}, We need your decision on the project proposal by {day}. The budget of {amount} has been approved pending your confirmation. Please respond ASAP.",
    "Dear {name}, Your annual performance review is scheduled for {day}. Please complete the self-assessment form and submit it to HR by {time} today.",
    "Hi {name}, The server migration is scheduled for {day} night. You are required to backup all critical data before {time}. This is a high-priority task from the CTO.",
    "Dear {name}, Your visa application requires additional documents. Please submit your bank statement, employment letter, and proof of residence by {day}. Deadline is strict.",
    "Hi {name}, The board meeting presentation needs to be ready by {day}. Please finalize the slides covering Q{q} revenue, growth metrics, and the roadmap for next year.",
    "IMPORTANT: {name}, Your contract renewal expires on {day}. Please review and sign the attached agreement. Contact legal at {time} for any clarifications.",
    "Dear {name}, The client has raised an urgent issue with the project deliverable. We need a hotfix deployed by {time} today. The manager and {team} team are standing by.",
    "Hi {name}, Your medical insurance renewal requires action by {day}. Please review your policy documents and confirm the premium payment of {amount}.",
    "Subject: Immediate Action Required - {name}, The production system is down. All engineers must report to the war room by {time}. This is a P0 incident affecting {amount} users.",
    "Dear {name}, Please note the new company policy effective from {day}. All employees must complete the compliance training by the end of this week. Failure will result in disciplinary action.",
    "Hi {name}, Your project budget request of {amount} has been approved by the finance team. Please initiate the procurement process immediately. Deadline for PO submission is {day}.",
    "Dear {name}, The quarterly audit starts on {day}. Please ensure all financial records, invoices, and receipts are organized and accessible by {time}.",
    "Hi {name}, I am writing to inform you that the company merger announcement will be made on {day}. You are required to maintain strict confidentiality until then.",
    "URGENT {name}: Your login credentials have been flagged for suspicious activity. Please change your password immediately and report to the IT security team by {time}.",
    "Hi {name}, The product launch is confirmed for {day}. All teams must complete final testing, documentation, and marketing materials before {time}. The CEO will be present.",
    "Dear {name}, Your probation period ends on {day}. A formal review meeting has been scheduled. Please prepare a summary of your achievements and learnings.",
    "Hi {name}, The client payment of {amount} is overdue by 30 days. Please escalate this to the finance manager and initiate the recovery process immediately.",
    "Dear {name}, The data center relocation requires your immediate input on {day}. Please finalize the hardware inventory and migration timeline with the {team} team.",
    "Hi {name}, This is a final notice for your pending tax filing. The deadline is {day} and penalties will apply. Please contact your tax advisor immediately.",
]

SPAM_EMAILS = [
    "Congratulations! You have been selected as our lucky winner! Claim your prize of $10,000 NOW! Click here to get your reward. Limited time offer. Don't miss out!",
    "FREE MONEY!!! We transfer millions of dollars per month. Work from home and earn $5000 daily. No experience needed. Join now and start earning today!!!",
    "URGENT: Your bank account has been compromised. Click this link immediately to verify your identity and secure your account before it is permanently locked.",
    "You won a FREE iPhone 15 Pro Max! You are the 1,000,000th visitor to our site! Claim your prize instantly. No credit card required. Click now before it expires!",
    "Make $10,000 per week working from home! Our proven system has helped thousands. No investment required. Sign up today and get rich quick! 100% guaranteed.",
    "HOT SINGLES IN YOUR AREA are waiting to meet you! Click here to see their profiles. Free registration for a limited time. Join millions of happy members today!",
    "WEIGHT LOSS MIRACLE PILL! Lose 30 pounds in 30 days guaranteed! Doctors HATE this secret. Buy 2 get 1 FREE. Order now and transform your body overnight!",
    "Your PayPal account is suspended! Verify your information immediately to avoid permanent closure. Click here and enter your details to restore your account.",
    "Nigerian Prince needs your help! I have $25 million USD I need to transfer urgently. You will receive 30% commission. Please reply with your bank details ASAP.",
    "Earn $500 per day just by clicking ads online! No skills required. Work from home in your pajamas. Thousands are doing it. Start your free trial today!!!",
    "FINAL NOTICE: You owe $4,999 in unpaid taxes. The IRS will issue a warrant for your arrest unless you pay immediately. Call this number right now to avoid jail.",
    "Buy Cheap Medications Online! Viagra, Cialis, and 100+ drugs without prescription! Lowest prices guaranteed! Discreet shipping worldwide. Order NOW!",
    "You are pre-approved for a $50,000 loan with no credit check! Get cash deposited to your account within 24 hours. Bad credit OK! Apply in 2 minutes.",
    "LIMITED OFFER: Get 1000 Instagram followers for just $5! Real and active followers guaranteed! Boost your profile and become an influencer today! Order now!",
    "CASINO WINNER ALERT! You have FREE spins waiting! Win real cash prizes playing online slots. No deposit required. Spin to win and cash out instantly!",
    "Cheap Rolex, Louis Vuitton, Gucci replica watches and bags. 99% identical to originals. Worldwide shipping available. Don't pay retail! Buy luxury for less.",
    "Act now! Your computer has been infected with 47 viruses! Download our FREE antivirus immediately to protect your data. Click here before it is too late!",
    "BITCOIN INVESTMENT OPPORTUNITY! Turn $500 into $50,000 in 30 days! Our AI trading bot guarantees 1000% returns. Join 100,000 successful investors today!",
    "FREE gift card worth $500 from Amazon, Walmart, or Target! Complete a short survey to claim yours. Only 100 cards remaining. Hurry before they run out!",
    "SECRET FOREX TRADING METHOD revealed! Our ex-banker made $2 million using this trick banks don't want you to know. Get the free ebook now!",
    "Congratulations! Your email was randomly selected in our monthly lottery! You have won 1,500,000 GBP. Contact our agent immediately to claim your winnings.",
    "HAIR REGROWTH GUARANTEED in 7 days! Our herbal formula works on baldness of all types. Buy 3 bottles get 2 free. 90-day money back guarantee. Order now!",
    "Your Netflix account has been charged incorrectly. Claim a $50 refund by clicking here and verifying your payment information. Process takes only 2 minutes.",
    "MLM OPPORTUNITY: Join our downline and earn passive income forever! $200 signup bonus. Make $5000 per month minimum. Work 2 hours per day. Life-changing income!",
    "FREE psychic reading! Our world-famous psychic will reveal your future, love life, and lucky numbers. Get your personal report for FREE. Reply with your birthdate!",
]

PROMOTIONAL_EMAILS = [
    "Flash Sale! Up to 70% off on all electronics this weekend only at TechZone. Shop now and save big on laptops, phones, tablets and accessories. Free shipping on orders above $50.",
    "Hey {name}! Your favorite brands are on sale. Don't miss up to 50% discount at FashionHub. New arrivals added daily. Use code SAVE20 for extra 20% off your cart.",
    "Summer Collection is here! Explore 500+ new styles starting at just $15. Free returns on all orders. Shop the latest trends before they sell out. Limited stock available.",
    "Exclusive Member Offer: Get 3 months of Premium for the price of 1! Stream unlimited movies and music ad-free. Offer valid only for existing members. Activate before {day}.",
    "New restaurant alert! FoodieHub has 50+ new partners near you. Get 40% off your first 3 orders. Free delivery this weekend. Download the app and order now!",
    "It's your Lucky Day! Spin the wheel and win up to 80% off at StyleMart. Every spin wins a prize! Shop from 10,000+ products with same-day delivery available.",
    "{name}, we miss you! Come back to ShopEasy and enjoy a special welcome back offer of 30% off your next purchase. Valid on all categories. Expires in 48 hours.",
    "Mega Book Fair starts {day}! Over 1 million books up to 60% off. Fiction, non-fiction, academic, children's books and more. Free gift wrapping on orders above $30.",
    "Your dream vacation is now affordable! Book flights and hotels together and save up to 45%. Destinations include Bali, Paris, Dubai and 200+ more. Book by {day}.",
    "Fitness goals made easy! Get 6 months free gym membership when you subscribe to our annual plan. 150+ classes, personal training sessions included. Join today!",
    "Home Makeover Sale! Furniture, decor, and kitchen essentials up to 55% off. Buy now pay later available. Same week delivery in most cities. Shop the collection.",
    "Celebrate your birthday month with us! Enjoy 25% off all orders plus a free gift on purchases above $75. Offer auto-applied at checkout. Happy Birthday from ShopWorld!",
    "New Year, New You! Start 2024 right with our Health & Wellness mega sale. Fitness equipment, supplements, and yoga gear up to 40% off. Free nutrition guide included.",
    "Limited Time: Buy 2 get 1 free on ALL perfumes and cosmetics at BeautyPalace. Over 500 brands available. Free samples with every order. Offer ends Sunday midnight!",
    "Upgrade your ride! Auto accessories, car care, and gadgets up to 35% off at AutoMart. Free installation on select products. EMI options available. Shop now!",
    "Kids Back to School Sale! Stationery, bags, uniforms, and electronics up to 50% off. Special bundle packs starting at $25. Fast delivery before school reopens!",
    "Premium Coffee Subscription! Get freshly roasted beans from 20+ origins delivered monthly. First box at 50% off. Cancel anytime. Your perfect morning awaits!",
    "Gaming Mega Sale! Latest titles up to 60% off, new console bundles, and accessories. Pre-order upcoming games at discounted price. GameZone newsletter exclusive.",
    "Handcrafted Jewelry Sale! Unique pieces in gold, silver, and gemstone. Prices starting at $20. Customization available. Perfect gifts for your loved ones. Shop now!",
    "Pet Lovers Special! Premium pet food, toys, and accessories at best prices. First order 30% off with free delivery. Vet-recommended brands. Your pet deserves the best!",
    "Online Course Sale! Master Python, Data Science, AI and more. Courses from $9.99. Lifetime access with certificate. Top instructors. Over 1 million students enrolled.",
    "Cloud Storage Deal! Get 2TB of secure cloud storage for just $2.99/month. Access from all devices. Share with family. Automatic backup. 3 months free trial today!",
    "Restaurant Week Special! Enjoy 3-course meals at top restaurants for fixed price $25. Participating restaurants in your city. Book your table now. Limited seats available!",
    "Travel Insurance now from $1/day! Covers medical emergencies, trip cancellation, lost baggage and more. Instant policy. 24/7 support. Get a quote in 2 minutes!",
    "Black Friday Preview! Get early access to deals before everyone else. Electronics, fashion, home goods - everything up to 80% off. Sign up to unlock preview access!",
]

SOCIAL_EMAILS = [
    "{name} sent you a friend request on ConnectHub. You may know them from college or work. Accept the request to stay connected and share life updates with each other.",
    "Your friend {name} just posted new photos from their vacation trip to Bali! Check out their latest updates and leave a comment. Don't miss what your friends are sharing!",
    "{name} mentioned you in a comment: 'Missing you @{name2}! We should catch up soon.' Reply to the conversation and see what others are saying about the post.",
    "You have 15 new notifications on SocialCircle! {name} and 14 others liked your recent post. See all your reactions, comments, and mentions in one place.",
    "Happy Birthday! Your friend {name} is celebrating their birthday today. Send them a warm wish and make their day special. Click here to write on their timeline!",
    "Alumni Update: {name} from your batch at {college} just updated their profile. They are now working as {role} at {company}. Connect and reconnect with old friends!",
    "New Group Invitation! {name} has added you to the group 'Class of 2024 Reunion'. 47 members are already in the group. Join the conversation and plan your meetup!",
    "{name} shared an article you might like: 'Top 10 AI Trends of 2024'. Your friends are talking about it. Read the article and share your thoughts in the comments.",
    "You have a new match on ConnectPro! {name} also liked your profile. You both work in the tech industry. Start a conversation now before someone else does!",
    "Event Invitation: {name} is hosting 'Annual Batch Reunion 2024' on {day}. 32 people are attending. RSVP by {day2} to secure your spot. Food and music included!",
    "Your LinkedIn connection {name} just published an article: 'How I landed my dream job at Google'. 2,500 people have read it. Add your thoughts to the discussion.",
    "Memory Alert: 3 years ago today you posted a photo with {name} and 4 others. Relive this beautiful memory and share it with your friends and family!",
    "New follower! {name} from {city} started following your photography account. Your latest post got 340 likes. Growing community loves your creative content!",
    "Your story was viewed by {name} and 127 others in the last 24 hours! Your poll results are in: 78% voted YES. Check the full analytics on your creator dashboard.",
    "Gaming Squad Alert! {name} is online and sent you a game request. Join them in Arena Legends for a multiplayer match. Your squad is waiting. Let's game!",
    "{name} commented on your photo: 'This is absolutely stunning! Where was this taken?' 25 people have liked this comment. Respond and keep the conversation going!",
    "Community Update: Your post in the 'Machine Learning Enthusiasts' group got 89 upvotes! You are this week's top contributor. Badges and rewards await you!",
    "New meetup in your area! 'Tech Professionals Networking Event' - 156 members from your field are attending on {day}. Register free and expand your professional network!",
    "Your Spotify Wrapped is ready! You listened to 85,000 minutes of music this year. Your top artist was {artist} and your top song was played 347 times. See your stats!",
    "Throwback Thursday: Your profile photo from 5 years ago is getting reactions! {name} and 45 others are reacting to your old photos. See which memories are trending!",
    "Book Club Notification: {name} suggested you read 'Atomic Habits' this month. 12 members have already started. Join the discussion thread and share your reading progress.",
    "Live Stream Alert! {name} is going live right now. 2,340 viewers are watching. Don't miss the live cooking demo. Join the stream and interact in real time!",
    "New comment on your YouTube video: '{name}: This is the best explanation I have ever seen! Please make more videos on this topic.' Your channel grew by 500 subscribers!",
    "Volunteer Opportunity: {name} and your college community are organizing a tree plantation drive on {day}. 85 volunteers have signed up. Join and make a difference!",
    "Friend Milestone: {name} just celebrated their 5-year work anniversary at {company}! Congratulate them on this achievement and let them know you care about their journey.",
]

UPDATES_EMAILS = [
    "Your order #ORD{order} has been shipped! Your package is on its way and will be delivered by {day}. Track your shipment in real-time using the TrackNow app. Thank you!",
    "Security Alert: A new login to your account was detected from {city} on {day} at {time}. If this was not you, please reset your password immediately and enable 2FA.",
    "Your monthly bank statement for {month} is now available. Total credits: {amount1}, Total debits: {amount2}. Log in to your banking app to view the full statement.",
    "App Update Available: Version 4.2.1 of {app} is now available. New features include dark mode, improved performance, and 15 bug fixes. Update now for the best experience.",
    "Flight Reminder: Your flight {flight} from {city1} to {city2} departs on {day} at {time}. Please arrive at the airport 2 hours early. Check-in closes 45 minutes before departure.",
    "Your subscription to {service} renews on {day}. Your credit card ending in {card} will be charged {amount}. To cancel or modify, visit account settings before the renewal date.",
    "System Maintenance Notice: Our services will be unavailable on {day} from {time1} to {time2} for scheduled maintenance. Please plan your work accordingly. We apologize for the inconvenience.",
    "Your refund of {amount} has been processed for order #ORD{order}. The amount will be credited to your original payment method within 5-7 business days. No further action needed.",
    "Password changed successfully! Your account password was updated on {day} at {time}. If you did not make this change, contact our support team immediately at support@service.com.",
    "New Policy Update: Our Terms of Service and Privacy Policy have been updated effective {day}. Key changes include data retention policy and third-party sharing rules. Please review.",
    "Your cloud backup completed successfully at {time}. Total data backed up: {size} GB. Next scheduled backup: {day}. All your important files are safe and encrypted.",
    "Delivery Attempted: We tried to deliver your package #ORD{order} but no one was available. Your package is now at the local facility. Reschedule delivery or pick up by {day}.",
    "Your electricity bill for {month} is ready. Amount due: {amount}. Due date: {day}. Auto-pay is enabled on your account. Visit the portal for detailed consumption analysis.",
    "Two-Factor Authentication Enabled: 2FA has been successfully activated on your account. Your account is now more secure. Backup codes have been generated. Save them safely.",
    "Software License Expiry Notice: Your {software} license expires in 15 days on {day}. Renew now to continue using all features without interruption. Renewal options available online.",
    "Payroll Processed: Your salary for {month} has been credited to your bank account ending {card}. Net pay: {amount}. View your detailed payslip in the HR portal.",
    "Your domain {domain} will expire on {day}. Renew now to prevent website downtime and losing your online presence. Auto-renewal is currently disabled for this domain.",
    "Video Upload Complete: Your video '{title}' has been processed and is now published on your channel. Duration: {duration} minutes. Share it with your audience now!",
    "Data Usage Alert: You have used 90% of your monthly mobile data plan. You have {size} MB remaining. Upgrade your plan to avoid speed throttling or extra charges.",
    "Meeting Recording Available: The recording of your meeting '{meeting}' from {day} is now available. Duration: {duration} minutes. Transcript and action items are also ready.",
    "Your annual tax summary for FY{year} is now available. Total income: {amount1}. Tax paid: {amount2}. Download Form 26AS and ITR acknowledgment from the portal.",
    "GitHub Repository Alert: A new pull request #{pr} has been opened in {repo}. Title: 'Fix critical authentication bug'. Review requested from {name}. Deadline: {day}.",
    "Your certificate for completing 'Advanced Machine Learning' course has been issued. Download it from your learning dashboard and add it to your LinkedIn profile.",
    "Internet Outage Resolved: The internet connectivity issue in your area has been resolved as of {time} on {day}. Services are fully restored. We apologize for the disruption.",
    "Your KYC verification has been completed successfully. Your account is now fully verified and all transaction limits have been upgraded. Thank you for your cooperation.",
]

# ── Helper function to fill templates ──
def fill_template(template):
    names = ["Rahul","Priya","Amit","Sneha","Raj","Kavya","Arjun","Meera","Vikram","Pooja"]
    days = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
    times = ["9:00 AM","10:30 AM","2:00 PM","4:30 PM","6:00 PM","11:59 PM"]
    amounts = ["$500","$1,200","$5,000","₹50,000","€2,500","$10,000","₹1,00,000"]
    docs = ["project","performance","audit","compliance","financial","technical"]
    teams = ["engineering","product","design","marketing","data science"]
    cities = ["New York","London","Mumbai","Bangalore","Tokyo","Dubai"]
    companies = ["Google","Microsoft","Amazon","TCS","Infosys","Wipro"]
    colleges = ["IIT Delhi","NIT Trichy","BITS Pilani","VIT","Anna University"]
    roles = ["Software Engineer","Data Scientist","Product Manager","AI Researcher"]
    services = ["Netflix","Spotify","Adobe","Microsoft 365","AWS"]
    apps = ["WhatsApp","Gmail","Zoom","Slack","VS Code"]
    months = ["January","February","March","April","May","June","July","August"]
    sizes = [str(random.randint(1,50))]
    orders = [str(random.randint(100000,999999))]
    cards = [str(random.randint(1000,9999))]
    artists = ["Taylor Swift","The Weeknd","Arijit Singh","Ed Sheeran"]
    titles = ["My Journey into AI","Travel Vlog 2024","Python Tutorial"]
    durations = [str(random.randint(20,120))]
    meetings = ["Sprint Review","All Hands Meeting","Design Review"]
    repos = ["backend-api","ml-pipeline","frontend-app"]
    domains = ["example.com","mysite.org","portfolio.dev"]
    prs = [str(random.randint(100,999))]
    flights = [f"AI{random.randint(100,999)}"]
    softwares = ["MATLAB","AutoCAD","Adobe Suite","JetBrains"]
    years = ["2023-24","2024-25"]
    outcomes = ["career","scholarship","promotion","bonus"]

    return template.format(
        name=random.choice(names),
        name2=random.choice(names),
        day=random.choice(days),
        day2=random.choice(days),
        time=random.choice(times),
        time1=random.choice(times),
        time2=random.choice(times),
        amount=random.choice(amounts),
        amount1=random.choice(amounts),
        amount2=random.choice(amounts),
        doc=random.choice(docs),
        q=random.randint(1,4),
        team=random.choice(teams),
        city=random.choice(cities),
        city1=random.choice(cities),
        city2=random.choice(cities),
        company=random.choice(companies),
        college=random.choice(colleges),
        role=random.choice(roles),
        service=random.choice(services),
        app=random.choice(apps),
        month=random.choice(months),
        size=random.choice(sizes),
        order=random.choice(orders),
        card=random.choice(cards),
        artist=random.choice(artists),
        title=random.choice(titles),
        duration=random.choice(durations),
        meeting=random.choice(meetings),
        repo=random.choice(repos),
        domain=random.choice(domains),
        pr=random.choice(prs),
        flight=random.choice(flights),
        software=random.choice(softwares),
        year=random.choice(years),
        outcome=random.choice(outcomes),
    )

# ── Build Dataset ──
random.seed(SEED)

def generate_samples(templates, label, n=200):
    samples = []
    for _ in range(n):
        t = random.choice(templates)
        try:
            text = fill_template(t)
        except (KeyError, IndexError):
            text = t
        samples.append({'text': text, 'label': label})
    return samples

data = []
data += generate_samples(IMPORTANT_EMAILS,   'Important',   250)
data += generate_samples(SPAM_EMAILS,        'Spam',        250)
data += generate_samples(PROMOTIONAL_EMAILS, 'Promotional', 200)
data += generate_samples(SOCIAL_EMAILS,      'Social',      200)
data += generate_samples(UPDATES_EMAILS,     'Updates',     200)

df = pd.DataFrame(data)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("="*65)
print("✅ DATASET GENERATED SUCCESSFULLY")
print("="*65)
print(f"📊 Total Samples   : {len(df)}")
print(f"📂 Categories      : {df['label'].nunique()}")
print("\n📈 Class Distribution:")
for label, count in df['label'].value_counts().items():
    pct = count/len(df)*100
    bar = '█' * int(pct/2)
    print(f"   {label:<15}: {count:>4} samples ({pct:.1f}%) {bar}")
print("="*65)
print("\n📧 Sample Emails:")
print("-"*65)
for i, row in df.sample(3, random_state=1).iterrows():
    print(f"[{row['label']}] {row['text'][:100]}...")
    print()

---
## 🧹 SECTION 3: NLP Preprocessing Pipeline
---

In [ ]:
# ============================================================
# CELL 4: Advanced NLP Preprocessing Pipeline
# ============================================================

print("🧹 Building NLP Preprocessing Pipeline...\n")

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# ── Custom Preprocessing ──
class EmailPreprocessor:
    """
    Advanced NLP Preprocessing Pipeline for Email Text.
    Steps: Clean → Lowercase → Tokenize → Remove Stopwords → Lemmatize
    """

    def __init__(self):
        self.stop_words = set(stopwords.words('english'))
        # Keep domain-specific keywords even if stopwords
        self.keep_words = {'not', 'no', 'never', 'against', 'urgent', 'free', 'win'}
        self.stop_words -= self.keep_words
        self.lemmatizer = WordNetLemmatizer()

    def clean_text(self, text):
        """Remove HTML tags, URLs, emails, special characters."""
        # Lowercase
        text = text.lower()
        # Remove HTML tags
        text = re.sub(r'<[^>]+>', ' ', text)
        # Remove URLs
        text = re.sub(r'http\S+|www\.\S+', ' URL ', text)
        # Remove email addresses
        text = re.sub(r'\S+@\S+', ' EMAIL ', text)
        # Remove phone numbers
        text = re.sub(r'\+?\d[\d\s\-().]{7,}\d', ' PHONE ', text)
        # Remove currency and numbers
        text = re.sub(r'[$€£₹]\s*\d+[,\d]*', ' MONEY ', text)
        text = re.sub(r'\b\d+\b', ' NUM ', text)
        # Remove special characters but keep spaces
        text = re.sub(r'[^a-z\s]', ' ', text)
        # Remove extra spaces
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def tokenize(self, text):
        """Tokenize the cleaned text."""
        return word_tokenize(text)

    def remove_stopwords(self, tokens):
        """Remove stopwords while preserving important domain terms."""
        return [t for t in tokens if t not in self.stop_words and len(t) > 1]

    def lemmatize(self, tokens):
        """Lemmatize tokens to their root forms."""
        return [self.lemmatizer.lemmatize(t, pos='v') for t in tokens]

    def preprocess(self, text):
        """Full preprocessing pipeline."""
        text = self.clean_text(text)
        tokens = self.tokenize(text)
        tokens = self.remove_stopwords(tokens)
        tokens = self.lemmatize(tokens)
        return ' '.join(tokens)

    def preprocess_batch(self, texts):
        """Batch preprocessing with progress bar."""
        return [self.preprocess(t) for t in tqdm(texts, desc="🔄 Preprocessing")]

# ── Apply Preprocessing ──
preprocessor = EmailPreprocessor()

print("📋 Step-by-step preprocessing example:")
print("-"*65)
sample_email = df['text'].iloc[0]
print(f"Original (first 120 chars):\n  {sample_email[:120]}...")
print()

# Show each step
step1 = preprocessor.clean_text(sample_email)
print(f"After Cleaning (first 120 chars):\n  {step1[:120]}...")

step2 = preprocessor.tokenize(step1)
print(f"\nAfter Tokenization (first 15 tokens):\n  {step2[:15]}")

step3 = preprocessor.remove_stopwords(step2)
print(f"\nAfter Stopword Removal (first 15 tokens):\n  {step3[:15]}")

step4 = preprocessor.lemmatize(step3)
print(f"\nAfter Lemmatization (first 15 tokens):\n  {step4[:15]}")

print("\n" + "="*65)
print("✅ Applying preprocessing to full dataset...")
df['processed_text'] = preprocessor.preprocess_batch(df['text'].tolist())

# Text Statistics
df['word_count_original']  = df['text'].apply(lambda x: len(x.split()))
df['word_count_processed'] = df['processed_text'].apply(lambda x: len(x.split()))
df['char_count']           = df['text'].apply(len)

print("\n📊 Text Statistics After Preprocessing:")
print("-"*65)
print(f"  Avg words (original)   : {df['word_count_original'].mean():.1f}")
print(f"  Avg words (processed)  : {df['word_count_processed'].mean():.1f}")
reduction = (1 - df['word_count_processed'].mean()/df['word_count_original'].mean())*100
print(f"  Vocabulary reduction   : {reduction:.1f}%")
print("="*65)
print("✅ Preprocessing Complete!")

---
## 🔢 SECTION 4: Feature Engineering & TF-IDF Vectorization
---

In [ ]:
# ============================================================
# CELL 5: TF-IDF Feature Extraction
# ============================================================

print("🔢 Building TF-IDF Feature Vectors...\n")

# ── Label Encoding ──
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])

CATEGORIES = label_encoder.classes_
print(f"📂 Categories: {list(CATEGORIES)}")
print(f"🔢 Encoding  : {dict(zip(CATEGORIES, range(len(CATEGORIES))))}\n")

# ── Train/Test Split ──
X_raw   = df['text'].values
X_proc  = df['processed_text'].values
y       = df['label_encoded'].values
y_names = df['label'].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train_proc, X_test_proc, _, _ = train_test_split(
    X_proc, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"📊 Train Set : {len(X_train_raw)} samples")
print(f"📊 Test Set  : {len(X_test_raw)} samples")

# ── TF-IDF Vectorizer (unigrams + bigrams) ──
tfidf_vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),          # Unigrams and Bigrams
    sublinear_tf=True,           # Apply log normalization
    min_df=2,                    # Minimum document frequency
    max_df=0.95,                 # Ignore too common terms
    analyzer='word',
    strip_accents='unicode',
    token_pattern=r'\b[a-zA-Z][a-zA-Z]+\b'
)

# Fit on training data only (prevent data leakage)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_proc)
X_test_tfidf  = tfidf_vectorizer.transform(X_test_proc)

print(f"\n🔢 TF-IDF Feature Matrix:")
print(f"   Train Shape : {X_train_tfidf.shape}")
print(f"   Test Shape  : {X_test_tfidf.shape}")
print(f"   Vocabulary  : {len(tfidf_vectorizer.vocabulary_):,} terms")
print(f"   Sparsity    : {(1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0]*X_train_tfidf.shape[1]))*100:.2f}%")

# Top Terms per Category
print("\n📋 Top TF-IDF Terms per Category:")
print("-"*65)
feature_names = tfidf_vectorizer.get_feature_names_out()

# Fit on all data to get category-level terms for display only
tfidf_all = TfidfVectorizer(max_features=15000, ngram_range=(1,2), sublinear_tf=True,
                             min_df=2, token_pattern=r'\b[a-zA-Z][a-zA-Z]+\b')
X_all_tfidf = tfidf_all.fit_transform(df['processed_text'])
all_features = tfidf_all.get_feature_names_out()

for label in CATEGORIES:
    indices = df[df['label'] == label].index
    cat_matrix = X_all_tfidf[indices]
    mean_tfidf = cat_matrix.mean(axis=0).A1
    top_idx = mean_tfidf.argsort()[-8:][::-1]
    top_terms = [all_features[i] for i in top_idx]
    print(f"  {label:<15}: {', '.join(top_terms)}")

print("="*65)
print("✅ Feature Extraction Complete!")

---
## 🤖 SECTION 5: Machine Learning Model Training
---

In [ ]:
# ============================================================
# CELL 6: Train & Evaluate Multiple ML Models
# ============================================================

print("🤖 Training Machine Learning Models...\n")
print("="*65)

# ── Model Definitions ──
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, C=1.0, solver='lbfgs',
        multi_class='multinomial', random_state=SEED
    ),
    'Naive Bayes': MultinomialNB(alpha=0.1),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=None, min_samples_split=2,
        random_state=SEED, n_jobs=-1
    ),
    'Linear SVM': LinearSVC(
        C=1.0, max_iter=2000, random_state=SEED
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1,
        max_depth=5, random_state=SEED
    ),
}

# ── Training & Evaluation ──
results = {}
trained_models = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for name, model in models.items():
    print(f"\n⚙️  Training: {name}")
    start = time.time()

    # Train
    model.fit(X_train_tfidf, y_train)
    train_time = time.time() - start

    # Predict
    y_pred = model.predict(X_test_tfidf)
    y_train_pred = model.predict(X_train_tfidf)

    # Metrics
    test_acc   = accuracy_score(y_test, y_pred)
    train_acc  = accuracy_score(y_train, y_train_pred)
    f1_macro   = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    precision  = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall     = recall_score(y_test, y_pred, average='macro', zero_division=0)

    # Cross Validation Score
    cv_scores = cross_val_score(model, X_train_tfidf, y_train, cv=3, scoring='accuracy', n_jobs=-1)

    results[name] = {
        'Train Accuracy': train_acc,
        'Test Accuracy':  test_acc,
        'F1 Macro':       f1_macro,
        'F1 Weighted':    f1_weighted,
        'Precision':      precision,
        'Recall':         recall,
        'CV Mean':        cv_scores.mean(),
        'CV Std':         cv_scores.std(),
        'Train Time(s)':  train_time,
        'y_pred':         y_pred,
    }
    trained_models[name] = model

    print(f"   ✅ Test Accuracy  : {test_acc*100:.2f}%")
    print(f"   📈 F1-Score (W)   : {f1_weighted*100:.2f}%")
    print(f"   🔄 CV Score       : {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")
    print(f"   ⏱️  Train Time     : {train_time:.2f}s")

print("\n" + "="*65)
print("✅ All Models Trained Successfully!")
print("="*65)

In [ ]:
# ============================================================
# CELL 7: Comprehensive Model Comparison Table
# ============================================================

print("\n📊 COMPREHENSIVE MODEL COMPARISON TABLE")
print("="*90)

# Build results dataframe
results_df = pd.DataFrame({
    name: {
        'Train Acc (%)':   f"{v['Train Accuracy']*100:.2f}",
        'Test Acc (%)':    f"{v['Test Accuracy']*100:.2f}",
        'F1-Macro (%)':    f"{v['F1 Macro']*100:.2f}",
        'F1-Weighted (%)': f"{v['F1 Weighted']*100:.2f}",
        'Precision (%)':   f"{v['Precision']*100:.2f}",
        'Recall (%)':      f"{v['Recall']*100:.2f}",
        'CV Score (%)':    f"{v['CV Mean']*100:.2f}±{v['CV Std']*100:.2f}",
        'Train Time(s)':   f"{v['Train Time(s)']:.2f}",
    }
    for name, v in results.items()
}).T

print(results_df.to_string())
print("="*90)

# Best Model
best_model_name = max(results, key=lambda k: results[k]['Test Accuracy'])
best_acc = results[best_model_name]['Test Accuracy']
print(f"\n🏆 Best Model: {best_model_name} with Test Accuracy = {best_acc*100:.2f}%")

# Detailed Report for Best Model
print(f"\n📋 Detailed Classification Report - {best_model_name}")
print("-"*65)
best_preds = results[best_model_name]['y_pred']
print(classification_report(y_test, best_preds, target_names=CATEGORIES))

---
## 🧠 SECTION 6: Deep Learning — BERT-based Classifier
---

In [ ]:
# ============================================================
# CELL 8: BERT-Based Email Classifier using DistilBERT
# ============================================================

print("🧠 Building BERT-based Classifier...")
print("   Using: distilbert-base-uncased (lightweight, fast)")
print("   ⚠️  GPU strongly recommended for training!\n")

BERT_MODEL_NAME = 'distilbert-base-uncased'
NUM_LABELS = len(CATEGORIES)
MAX_LENGTH = 128
BERT_BATCH_SIZE = 16
BERT_EPOCHS = 2   # Increase to 3-4 for better accuracy

# ── HuggingFace Dataset Class ──
class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ── Load Tokenizer & Model ──
print("⬇️  Loading DistilBERT tokenizer and model...")
bert_tokenizer = DistilBertTokenizer.from_pretrained(BERT_MODEL_NAME)
bert_model     = DistilBertForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME, num_labels=NUM_LABELS
).to(device)

print(f"✅ Model loaded | Parameters: {sum(p.numel() for p in bert_model.parameters()):,}")

# ── Create Datasets ──
# Use a smaller subset for faster training in Colab
N_TRAIN  = min(600, len(X_train_raw))
N_TEST   = min(200, len(X_test_raw))

train_dataset = EmailDataset(
    list(X_train_raw[:N_TRAIN]), list(y_train[:N_TRAIN]),
    bert_tokenizer, MAX_LENGTH
)
test_dataset  = EmailDataset(
    list(X_test_raw[:N_TEST]), list(y_test[:N_TEST]),
    bert_tokenizer, MAX_LENGTH
)

train_loader = DataLoader(train_dataset, batch_size=BERT_BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BERT_BATCH_SIZE, shuffle=False)

print(f"\n📊 BERT Dataset:")
print(f"   Train : {N_TRAIN} samples → {len(train_loader)} batches")
print(f"   Test  : {N_TEST} samples → {len(test_loader)} batches")

# ── Fine-tuning Loop ──
optimizer  = torch.optim.AdamW(bert_model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * BERT_EPOCHS
scheduler  = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1.0, end_factor=0.1, total_iters=total_steps
)

bert_train_losses = []
bert_val_accs     = []

print(f"\n🚂 Starting BERT Fine-tuning ({BERT_EPOCHS} epochs)...")
print("-"*65)

for epoch in range(BERT_EPOCHS):
    bert_model.train()
    total_loss = 0
    correct = 0
    total   = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{BERT_EPOCHS}")
    for batch in pbar:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        logits  = outputs.logits

        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{correct/total:.4f}'})

    avg_loss   = total_loss / len(train_loader)
    train_acc  = correct / total
    bert_train_losses.append(avg_loss)

    # Validation
    bert_model.eval()
    val_correct = 0
    val_total   = 0
    bert_preds_list = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs  = bert_model(input_ids=input_ids, attention_mask=attention_mask)
            preds    = outputs.logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)
            bert_preds_list.extend(preds.cpu().numpy())

    val_acc = val_correct / val_total
    bert_val_accs.append(val_acc)

    print(f"\n📊 Epoch {epoch+1}: Loss={avg_loss:.4f} | Train Acc={train_acc*100:.2f}% | Val Acc={val_acc*100:.2f}%")

# Final BERT Metrics
bert_final_acc = bert_val_accs[-1]
bert_preds_np  = np.array(bert_preds_list)
bert_f1_w      = f1_score(y_test[:N_TEST], bert_preds_np, average='weighted')

print("\n" + "="*65)
print("✅ BERT Fine-tuning Complete!")
print(f"   🎯 Final Val Accuracy : {bert_final_acc*100:.2f}%")
print(f"   📈 F1-Score Weighted  : {bert_f1_w*100:.2f}%")
print("="*65)

In [ ]:
# ============================================================
# CELL 9: BERT vs Traditional ML Comparison
# ============================================================

print("📊 BERT vs Traditional ML — Final Comparison")
print("="*65)

comparison_data = []
for name, res in results.items():
    comparison_data.append({
        'Model':         name,
        'Type':          'Traditional ML',
        'Test Acc (%)':  round(res['Test Accuracy']*100, 2),
        'F1-Weighted':   round(res['F1 Weighted']*100, 2),
        'Speed':         'Fast',
        'Interpretable': 'Yes'
    })

comparison_data.append({
    'Model':         'DistilBERT',
    'Type':          'Deep Learning (Transformer)',
    'Test Acc (%)':  round(bert_final_acc*100, 2),
    'F1-Weighted':   round(bert_f1_w*100, 2),
    'Speed':         'Slow (GPU)',
    'Interpretable': 'Limited'
})

comp_df = pd.DataFrame(comparison_data)
comp_df = comp_df.sort_values('Test Acc (%)', ascending=False).reset_index(drop=True)
comp_df.index = comp_df.index + 1

print(comp_df.to_string())
print("\n" + "="*65)

print("\n💡 Key Insights:")
print("  • BERT leverages contextual embeddings for richer representations.")
print("  • Traditional ML (esp. SVM/LR) is faster and surprisingly competitive.")
print("  • For production, ensemble or distilled BERT models balance trade-offs.")
print("  • BERT needs GPU and more data to show its full potential.")

---
## 🎯 SECTION 7: Priority Scoring System
---

In [ ]:
# ============================================================
# CELL 10: Email Priority Scoring System (0-100)
# ============================================================

print("🎯 Building Email Priority Scoring System...\n")

class PriorityScorer:
    """
    Multi-factor Email Priority Scoring System.
    Score ranges from 0 (low priority) to 100 (critical priority).
    """

    # ── Urgency Keyword Dictionary with Weights ──
    URGENCY_KEYWORDS = {
        # Tier 1 — Critical (weight: 20)
        'urgent': 20, 'emergency': 20, 'critical': 20, 'immediate': 18,
        'asap': 18, 'immediately': 17, 'p0': 20, 'incident': 18,

        # Tier 2 — High Priority (weight: 15)
        'deadline': 15, 'due': 12, 'overdue': 15, 'payment': 14,
        'invoice': 13, 'final notice': 15, 'expire': 14, 'expiry': 14,
        'last chance': 15, 'action required': 15, 'mandatory': 15,

        # Tier 3 — Work/Professional (weight: 12)
        'interview': 12, 'project': 10, 'manager': 11, 'meeting': 10,
        'ceo': 13, 'director': 12, 'board': 12, 'client': 11,
        'contract': 12, 'approval': 12, 'review': 9, 'report': 8,

        # Tier 4 — Time-Sensitive (weight: 8)
        'today': 10, 'tonight': 10, 'tomorrow': 8, 'this week': 6,
        'by end of': 10, 'before': 7, 'within': 6, 'hours': 8,

        # Tier 5 — Financial (weight: 10)
        'salary': 10, 'bonus': 9, 'tax': 10, 'audit': 11, 'budget': 9,
        'refund': 9, 'billing': 10, 'account': 8, 'bank': 10,

        # Tier 6 — Security (weight: 12)
        'password': 12, 'security': 12, 'breach': 15, 'hack': 15,
        'suspicious': 12, 'compromised': 15, 'verify': 10,
    }

    CATEGORY_BASE_SCORES = {
        'Important':   70,
        'Updates':     40,
        'Social':      20,
        'Promotional': 15,
        'Spam':         5,
    }

    def compute_urgency_score(self, text):
        """Compute urgency based on keyword weights."""
        text_lower = text.lower()
        score = 0
        matched = []
        for keyword, weight in self.URGENCY_KEYWORDS.items():
            if keyword in text_lower:
                score += weight
                matched.append((keyword, weight))
        return min(score, 50), matched  # Cap urgency at 50

    def compute_length_score(self, text):
        """Longer emails from professionals tend to be important."""
        word_count = len(text.split())
        if word_count > 100: return 8
        elif word_count > 50: return 5
        elif word_count > 20: return 3
        return 0

    def compute_capitalization_score(self, text):
        """ALLCAPS often signals urgency (or spam)."""
        caps_ratio = sum(1 for c in text if c.isupper()) / max(len(text), 1)
        if caps_ratio > 0.3: return 5    # Many caps
        elif caps_ratio > 0.15: return 3
        return 0

    def compute_exclamation_score(self, text):
        """Exclamation marks indicate urgency or spam."""
        count = text.count('!')
        return min(count * 2, 8)

    def compute_priority_score(self, text, category):
        """Compute composite priority score 0-100."""
        # Base score from category prediction
        base = self.CATEGORY_BASE_SCORES.get(category, 30)

        # Urgency keywords
        urgency_score, matched = self.compute_urgency_score(text)

        # Additional signals
        length_score   = self.compute_length_score(text)
        caps_score     = self.compute_capitalization_score(text)
        exclaim_score  = self.compute_exclamation_score(text)

        # Spam penalty: reduce priority
        spam_penalty = -20 if category == 'Spam' else 0

        # Composite score
        raw_score = base + urgency_score + length_score + caps_score + exclaim_score + spam_penalty
        final_score = max(0, min(100, raw_score))

        return {
            'priority_score':  final_score,
            'base_score':      base,
            'urgency_score':   urgency_score,
            'length_score':    length_score,
            'caps_score':      caps_score,
            'exclaim_score':   exclaim_score,
            'spam_penalty':    spam_penalty,
            'matched_keywords': [kw for kw, _ in matched[:5]],
        }

    def get_priority_label(self, score):
        """Map numeric score to human-readable priority label."""
        if score >= 80: return '🔴 CRITICAL'
        elif score >= 60: return '🟠 HIGH'
        elif score >= 40: return '🟡 MEDIUM'
        elif score >= 20: return '🟢 LOW'
        return '⚪ NEGLIGIBLE'


priority_scorer = PriorityScorer()

# ── Apply to Dataset ──
print("⚙️  Computing priority scores for all emails...")

# Get best model predictions for full dataset
X_all_tfidf = tfidf_vectorizer.transform(df['processed_text'])
df['predicted_label'] = label_encoder.inverse_transform(
    trained_models[best_model_name].predict(X_all_tfidf)
)

priority_results = []
for _, row in df.iterrows():
    result = priority_scorer.compute_priority_score(
        row['text'], row['predicted_label']
    )
    priority_results.append(result['priority_score'])

df['priority_score'] = priority_results
df['priority_label'] = df['priority_score'].apply(priority_scorer.get_priority_label)

print("\n📊 Priority Score Statistics by Category:")
print("-"*65)
priority_stats = df.groupby('label')['priority_score'].agg(['mean','min','max','std'])
priority_stats.columns = ['Mean Score', 'Min Score', 'Max Score', 'Std Dev']
priority_stats = priority_stats.round(2)
print(priority_stats.to_string())

print("\n📊 Priority Level Distribution:")
print("-"*65)
for level in ['🔴 CRITICAL','🟠 HIGH','🟡 MEDIUM','🟢 LOW','⚪ NEGLIGIBLE']:
    count = (df['priority_label'] == level).sum()
    pct = count/len(df)*100
    print(f"  {level:<20}: {count:>4} emails ({pct:.1f}%)")

print("\n" + "="*65)
print("✅ Priority Scoring Complete!")

---
## 📝 SECTION 8: Email Summarization
---

In [ ]:
# ============================================================
# CELL 11: HuggingFace Transformer — Email Summarization
# ============================================================

print("📝 Loading Summarization Model (facebook/bart-large-cnn)...")
print("   ⚠️  First load may take 1-2 minutes.\n")

# ── Load Summarization Pipeline ──
try:
    summarizer = pipeline(
        'summarization',
        model='facebook/bart-large-cnn',
        device=0 if torch.cuda.is_available() else -1
    )
    SUMMARIZER_LOADED = True
    print("✅ Summarization model loaded successfully!")
except Exception as e:
    print(f"⚠️  Could not load BART, using fallback summarizer. Error: {e}")
    SUMMARIZER_LOADED = False

# ── Fallback Extractive Summarizer ──
class ExtractiveSummarizer:
    """
    TF-IDF based extractive summarizer as fallback.
    Selects top N sentences based on importance score.
    """
    def __init__(self, num_sentences=2):
        self.num_sentences = num_sentences

    def summarize(self, text):
        sentences = sent_tokenize(text)
        if len(sentences) <= self.num_sentences:
            return text
        # Score sentences by word importance (simple TF)
        word_freq = Counter(text.lower().split())
        scores = []
        for sent in sentences:
            score = sum(word_freq[w.lower()] for w in sent.split())
            scores.append((score / max(len(sent.split()), 1), sent))
        scores.sort(reverse=True)
        top_sents = [s[1] for s in scores[:self.num_sentences]]
        # Restore original order
        ordered = [s for s in sentences if s in top_sents]
        return ' '.join(ordered)


extractive_summarizer = ExtractiveSummarizer(num_sentences=2)


def generate_summary(text, use_transformer=True):
    """
    Generate AI summary of email.
    Uses BART if available, else extractive summarizer.
    """
    if len(text.split()) < 30:
        return text  # Short emails don't need summarization

    if SUMMARIZER_LOADED and use_transformer:
        try:
            result = summarizer(
                text[:1024],  # BART max length
                max_length=60,
                min_length=20,
                do_sample=False
            )
            return result[0]['summary_text']
        except Exception:
            pass

    return extractive_summarizer.summarize(text)


# ── Demo: Summarize Sample Emails ──
print("\n📧 Email Summarization Demo")
print("="*65)

sample_emails_for_summary = [
    {
        'text': """Dear Rahul, I am writing to inform you that your project submission deadline has been extended by one week. However, this is the final extension that will be granted. The complete project including code, documentation, testing report, and presentation slides must be submitted to the department portal before 11:59 PM on Friday. The project will be evaluated by a panel of three faculty members and an industry expert from TCS. Please ensure your code is properly commented, all edge cases are handled, and the README file is complete. Failure to submit on time will result in a zero grade for the project component, which carries 40% weightage in your final grade. If you face any technical issues, contact the lab assistant before Thursday evening. Best regards, Prof. Sharma, Department of Computer Science""",
        'label': 'Important'
    },
    {
        'text': """CONGRATULATIONS! You have been selected as our GRAND PRIZE WINNER in the International Lucky Draw 2024! You have won a brand new BMW X5, $50,000 in cash, and a luxury vacation package to Maldives! To claim your prize, you need to send us a processing fee of just $299 via Western Union. This is completely refundable after prize delivery. Your prize reference number is LUCKY2024-9876. You have been selected from over 10 million email addresses worldwide! Please respond within 48 hours or your prize will be forfeited. Contact agent James Wilson at claimyourprize@freemail.com. Do not tell anyone as this is a confidential offer!""",
        'label': 'Spam'
    },
    {
        'text': """Hi valued customer! The biggest sale of the year is here at TechWorld! Starting from midnight tonight, enjoy incredible discounts on all our top products. Laptops from leading brands like Dell, HP, and Lenovo are available at up to 45% off. Mobile phones including iPhone 15 series and Samsung Galaxy S24 are priced at their lowest ever. Smart TVs, tablets, earphones, and gaming accessories are also part of this massive sale. Use coupon code MEGA45 at checkout to get an additional 10% off on already discounted items. Free delivery on all orders above $100. EMI options available with zero interest for 12 months. Sale ends Sunday midnight. Shop now at www.techworldsale.com""",
        'label': 'Promotional'
    },
]

for i, item in enumerate(sample_emails_for_summary, 1):
    print(f"\n📧 Email {i} [{item['label']}]")
    print(f"Original ({len(item['text'].split())} words):")
    print(f"  {item['text'][:200]}...")

    summary = generate_summary(item['text'])
    print(f"\n🤖 AI Summary ({len(summary.split())} words):")
    print(f"  {summary}")
    print("-"*65)

print("\n✅ Summarization Module Ready!")

---
## 📊 SECTION 9: Visualization Dashboard
---

In [ ]:
# ============================================================
# CELL 12: Comprehensive Analytics Dashboard
# ============================================================

print("📊 Generating Analytics Dashboard...\n")

# ── Color Palette ──
COLORS = {
    'Important':   '#E63946',
    'Spam':        '#F4A261',
    'Promotional': '#2A9D8F',
    'Social':      '#457B9D',
    'Updates':     '#8338EC'
}
COLOR_LIST = list(COLORS.values())

fig = plt.figure(figsize=(22, 28))
fig.patch.set_facecolor('#0A0E1A')
gs = GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

TITLE_COLOR = 'white'
LABEL_COLOR = '#CCCCCC'
GRID_COLOR  = '#1E2A3A'

def style_ax(ax, title):
    ax.set_facecolor('#0F1620')
    ax.set_title(title, color=TITLE_COLOR, fontsize=12, fontweight='bold', pad=10)
    ax.tick_params(colors=LABEL_COLOR)
    ax.xaxis.label.set_color(LABEL_COLOR)
    ax.yaxis.label.set_color(LABEL_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID_COLOR)
    ax.grid(color=GRID_COLOR, linestyle='--', linewidth=0.5, alpha=0.7)

# ── Plot 1: Category Distribution (Donut) ──
ax1 = fig.add_subplot(gs[0, 0])
counts = df['label'].value_counts()
wedges, texts, autotexts = ax1.pie(
    counts.values, labels=counts.index, autopct='%1.1f%%',
    colors=[COLORS[k] for k in counts.index],
    wedgeprops=dict(width=0.6, edgecolor='#0A0E1A', linewidth=2),
    startangle=90
)
for t in texts: t.set_color(LABEL_COLOR); t.set_fontsize(9)
for at in autotexts: at.set_color('white'); at.set_fontsize(8); at.set_fontweight('bold')
ax1.set_facecolor('#0F1620')
ax1.set_title('📂 Email Category Distribution', color=TITLE_COLOR, fontsize=12, fontweight='bold')

# ── Plot 2: Accuracy Comparison (Horizontal Bar) ──
ax2 = fig.add_subplot(gs[0, 1])
model_names = list(results.keys()) + ['DistilBERT']
accuracies  = [results[m]['Test Accuracy']*100 for m in results] + [bert_final_acc*100]
bar_colors  = ['#E63946','#F4A261','#2A9D8F','#457B9D','#8338EC','#FFD700']
bars = ax2.barh(model_names, accuracies, color=bar_colors, edgecolor='none', height=0.6)
for bar, acc in zip(bars, accuracies):
    ax2.text(acc + 0.3, bar.get_y() + bar.get_height()/2,
             f'{acc:.1f}%', va='center', color='white', fontsize=9, fontweight='bold')
ax2.set_xlim(0, 110)
ax2.set_xlabel('Accuracy (%)', color=LABEL_COLOR)
style_ax(ax2, '🏆 Model Accuracy Comparison')

# ── Plot 3: Priority Score Distribution ──
ax3 = fig.add_subplot(gs[0, 2])
for label in CATEGORIES:
    subset = df[df['label'] == label]['priority_score']
    ax3.hist(subset, bins=20, alpha=0.6, label=label, color=COLORS[label], edgecolor='none')
ax3.set_xlabel('Priority Score (0-100)', color=LABEL_COLOR)
ax3.set_ylabel('Count', color=LABEL_COLOR)
legend = ax3.legend(fontsize=8, facecolor='#0F1620', edgecolor=GRID_COLOR, labelcolor=LABEL_COLOR)
style_ax(ax3, '🎯 Priority Score Distribution')

# ── Plot 4: F1-Score Comparison ──
ax4 = fig.add_subplot(gs[1, 0])
metrics_names = ['F1-Macro', 'F1-Weighted', 'Precision', 'Recall']
x_pos = np.arange(len(metrics_names))
width = 0.15
plot_models = list(results.keys())[:4]
model_colors2 = ['#E63946','#F4A261','#2A9D8F','#457B9D']
for i, (mname, mcol) in enumerate(zip(plot_models, model_colors2)):
    vals = [
        results[mname]['F1 Macro']*100,
        results[mname]['F1 Weighted']*100,
        results[mname]['Precision']*100,
        results[mname]['Recall']*100,
    ]
    ax4.bar(x_pos + i*width, vals, width, label=mname, color=mcol, alpha=0.9)
ax4.set_xticks(x_pos + width * 1.5)
ax4.set_xticklabels(metrics_names, rotation=15, color=LABEL_COLOR, fontsize=8)
ax4.set_ylabel('Score (%)', color=LABEL_COLOR)
ax4.set_ylim(0, 115)
legend = ax4.legend(fontsize=7, facecolor='#0F1620', edgecolor=GRID_COLOR, labelcolor=LABEL_COLOR)
style_ax(ax4, '📈 Multi-Metric Comparison')

# ── Plot 5: Spam vs Important vs Others ──
ax5 = fig.add_subplot(gs[1, 1])
spam_count     = (df['label'] == 'Spam').sum()
imp_count      = (df['label'] == 'Important').sum()
other_count    = len(df) - spam_count - imp_count
categories_pie = ['Spam', 'Important', 'Others']
values_pie     = [spam_count, imp_count, other_count]
colors_pie     = ['#F4A261', '#E63946', '#8338EC']
explode        = (0.05, 0.05, 0)
wedges2, texts2, autotexts2 = ax5.pie(
    values_pie, labels=categories_pie, autopct='%1.1f%%',
    colors=colors_pie, explode=explode,
    wedgeprops=dict(edgecolor='#0A0E1A', linewidth=2), startangle=45
)
for t in texts2: t.set_color(LABEL_COLOR)
for at in autotexts2: at.set_color('white'); at.set_fontweight('bold')
ax5.set_facecolor('#0F1620')
ax5.set_title('⚠️ Spam vs Important vs Others', color=TITLE_COLOR, fontsize=12, fontweight='bold')

# ── Plot 6: Priority Score by Category (Boxplot) ──
ax6 = fig.add_subplot(gs[1, 2])
data_for_box = [df[df['label'] == label]['priority_score'].values for label in CATEGORIES]
bp = ax6.boxplot(data_for_box, patch_artist=True, notch=True, labels=CATEGORIES,
                 medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], COLOR_LIST):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
for whisker in bp['whiskers']: whisker.set_color(LABEL_COLOR)
for cap in bp['caps']: cap.set_color(LABEL_COLOR)
for flier in bp['fliers']:
    flier.set(marker='o', color=LABEL_COLOR, markersize=4, alpha=0.5)
ax6.set_xticklabels(CATEGORIES, rotation=20, fontsize=8, color=LABEL_COLOR)
ax6.set_ylabel('Priority Score', color=LABEL_COLOR)
style_ax(ax6, '📦 Priority Score by Category')

# ── Plot 7: Word Count Distribution ──
ax7 = fig.add_subplot(gs[2, 0])
for label in CATEGORIES:
    subset = df[df['label'] == label]['word_count_original']
    ax7.hist(subset, bins=25, alpha=0.6, label=label, color=COLORS[label], density=True)
ax7.set_xlabel('Word Count', color=LABEL_COLOR)
ax7.set_ylabel('Density', color=LABEL_COLOR)
legend = ax7.legend(fontsize=8, facecolor='#0F1620', edgecolor=GRID_COLOR, labelcolor=LABEL_COLOR)
style_ax(ax7, '📏 Email Length Distribution')

# ── Plot 8: BERT Training Loss Curve ──
ax8 = fig.add_subplot(gs[2, 1])
if bert_train_losses:
    epochs_range = range(1, len(bert_train_losses)+1)
    ax8.plot(epochs_range, bert_train_losses, 'o-', color='#E63946', linewidth=2,
             markersize=8, label='Train Loss')
    ax8.plot(epochs_range, [1-v for v in bert_val_accs], 's--', color='#2A9D8F',
             linewidth=2, markersize=8, label='Val Error')
    ax8.set_xlabel('Epoch', color=LABEL_COLOR)
    ax8.set_ylabel('Loss / Error', color=LABEL_COLOR)
    ax8.set_xticks(epochs_range)
    legend = ax8.legend(fontsize=9, facecolor='#0F1620', edgecolor=GRID_COLOR, labelcolor=LABEL_COLOR)
style_ax(ax8, '🧠 BERT Training Curve')

# ── Plot 9: Model Training Time ──
ax9 = fig.add_subplot(gs[2, 2])
train_times = [results[m]['Train Time(s)'] for m in results]
times_colors = ['#E63946','#F4A261','#2A9D8F','#457B9D','#8338EC']
bars2 = ax9.bar(list(results.keys()), train_times, color=times_colors, edgecolor='none')
for bar, t in zip(bars2, train_times):
    ax9.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{t:.2f}s', ha='center', color='white', fontsize=9, fontweight='bold')
ax9.set_ylabel('Time (seconds)', color=LABEL_COLOR)
ax9.set_xticklabels(list(results.keys()), rotation=20, ha='right', fontsize=8, color=LABEL_COLOR)
style_ax(ax9, '⏱️ Training Time Comparison')

# ── Plot 10: CV Score with Error Bars ──
ax10 = fig.add_subplot(gs[3, 0])
cv_means = [results[m]['CV Mean']*100 for m in results]
cv_stds  = [results[m]['CV Std']*100 for m in results]
x_cv = np.arange(len(results))
ax10.bar(x_cv, cv_means, color=times_colors, alpha=0.8, edgecolor='none')
ax10.errorbar(x_cv, cv_means, yerr=cv_stds, fmt='none', color='white', capsize=6, linewidth=2)
ax10.set_xticks(x_cv)
ax10.set_xticklabels(list(results.keys()), rotation=20, ha='right', fontsize=8, color=LABEL_COLOR)
ax10.set_ylabel('CV Accuracy (%)', color=LABEL_COLOR)
ax10.set_ylim(0, 115)
style_ax(ax10, '🔄 Cross-Validation Scores (5-fold)')

# ── Plot 11: Priority Level Counts ──
ax11 = fig.add_subplot(gs[3, 1])
priority_levels  = ['🔴 CRITICAL','🟠 HIGH','🟡 MEDIUM','🟢 LOW','⚪ NEGLIGIBLE']
priority_counts  = [( df['priority_label'] == level).sum() for level in priority_levels]
priority_colors  = ['#E63946','#F4A261','#FFD700','#2A9D8F','#8338EC']
clean_labels     = ['CRITICAL','HIGH','MEDIUM','LOW','NEGLIGIBLE']
bars3 = ax11.bar(clean_labels, priority_counts, color=priority_colors, edgecolor='none')
for bar, cnt in zip(bars3, priority_counts):
    ax11.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
              str(cnt), ha='center', color='white', fontsize=10, fontweight='bold')
ax11.set_ylabel('Number of Emails', color=LABEL_COLOR)
ax11.set_xticklabels(clean_labels, rotation=15, color=LABEL_COLOR, fontsize=9)
style_ax(ax11, '🎯 Email Priority Level Breakdown')

# ── Plot 12: Heatmap — Category vs Priority Level ──
ax12 = fig.add_subplot(gs[3, 2])
heatmap_data = pd.crosstab(df['label'], df['priority_label'])
# Reorder columns
col_order = [c for c in priority_levels if c in heatmap_data.columns]
heatmap_data = heatmap_data.reindex(columns=col_order, fill_value=0)
heatmap_data.columns = ['CRITICAL','HIGH','MEDIUM','LOW','NEGLIGIBLE'][:len(col_order)]
im = ax12.imshow(heatmap_data.values, cmap='YlOrRd', aspect='auto')
ax12.set_xticks(range(len(heatmap_data.columns)))
ax12.set_yticks(range(len(heatmap_data.index)))
ax12.set_xticklabels(heatmap_data.columns, rotation=30, color=LABEL_COLOR, fontsize=8)
ax12.set_yticklabels(heatmap_data.index, color=LABEL_COLOR, fontsize=9)
for i in range(len(heatmap_data.index)):
    for j in range(len(heatmap_data.columns)):
        val = heatmap_data.values[i, j]
        ax12.text(j, i, str(val), ha='center', va='center',
                  color='black' if val > heatmap_data.values.max()*0.6 else 'white', fontsize=9)
plt.colorbar(im, ax=ax12)
ax12.set_facecolor('#0F1620')
ax12.set_title('🗺️ Category × Priority Heatmap', color=TITLE_COLOR, fontsize=12, fontweight='bold')

# ── Master Title ──
fig.suptitle(
    '🤖 AI-Powered Smart Email Management System — Analytics Dashboard',
    color='white', fontsize=16, fontweight='bold', y=0.98
)

plt.savefig('email_analytics_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor='#0A0E1A')
plt.show()
print("\n✅ Analytics Dashboard Generated & Saved!")

---
## 🔍 SECTION 10: Confusion Matrix Analysis
---

In [ ]:
# ============================================================
# CELL 13: Confusion Matrix for All Models
# ============================================================

print("🔍 Generating Confusion Matrices...\n")

fig, axes = plt.subplots(2, 3, figsize=(22, 13))
fig.patch.set_facecolor('#0A0E1A')
axes = axes.flatten()

# Include BERT
plot_models_cm = list(results.keys()) + ['DistilBERT']
all_preds = {name: results[name]['y_pred'] for name in results}
all_preds['DistilBERT'] = np.array(bert_preds_list)
all_ytrue = {name: y_test for name in results}
all_ytrue['DistilBERT'] = y_test[:N_TEST]

cmaps = ['Reds', 'Oranges', 'Greens', 'Blues', 'Purples', 'YlOrBr']

for idx, (model_name, cmap) in enumerate(zip(plot_models_cm, cmaps)):
    ax = axes[idx]
    y_true_cm = all_ytrue[model_name]
    y_pred_cm = all_preds[model_name]

    # Get unique labels in this test set
    unique_labels = sorted(set(y_true_cm))
    cat_labels = [CATEGORIES[i] for i in unique_labels]

    cm = confusion_matrix(y_true_cm, y_pred_cm, labels=unique_labels)
    cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    im = ax.imshow(cm_normalized, interpolation='nearest', cmap=cmap, vmin=0, vmax=1)

    # Annotations
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            text_color = 'white' if cm_normalized[i, j] < 0.6 else 'black'
            ax.text(j, i, f'{cm[i,j]}\n({cm_normalized[i,j]:.1%})',
                    ha='center', va='center', fontsize=8,
                    color=text_color, fontweight='bold')

    ax.set_facecolor('#0F1620')
    acc = accuracy_score(y_true_cm, y_pred_cm)
    ax.set_title(f'{model_name}\nAccuracy: {acc*100:.1f}%',
                 color='white', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted Label', color='#CCCCCC', fontsize=9)
    ax.set_ylabel('True Label', color='#CCCCCC', fontsize=9)
    ax.set_xticks(range(len(cat_labels)))
    ax.set_yticks(range(len(cat_labels)))
    ax.set_xticklabels(cat_labels, rotation=25, ha='right', fontsize=8, color='#CCCCCC')
    ax.set_yticklabels(cat_labels, fontsize=8, color='#CCCCCC')
    plt.colorbar(im, ax=ax)

fig.suptitle('🔍 Confusion Matrices — All Models',
             color='white', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight', facecolor='#0A0E1A')
plt.show()
print("✅ Confusion Matrices Generated!")

In [ ]:
# ============================================================
# CELL 14: Interactive Plotly Charts
# ============================================================

print("📊 Generating Interactive Plotly Charts...\n")

# ── 1. Interactive Accuracy Comparison ──
model_names_all = list(results.keys()) + ['DistilBERT']
acc_values = [results[m]['Test Accuracy']*100 for m in results] + [bert_final_acc*100]
f1_values  = [results[m]['F1 Weighted']*100 for m in results] + [bert_f1_w*100]

fig_plotly = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Model Accuracy Comparison', 'Category Priority Score Distribution'),
    horizontal_spacing=0.1
)

# Accuracy + F1
fig_plotly.add_trace(
    go.Bar(name='Test Accuracy', x=model_names_all, y=acc_values,
           marker_color='#E63946', text=[f'{v:.1f}%' for v in acc_values], textposition='outside'),
    row=1, col=1
)
fig_plotly.add_trace(
    go.Bar(name='F1-Weighted', x=model_names_all, y=f1_values,
           marker_color='#2A9D8F', text=[f'{v:.1f}%' for v in f1_values], textposition='outside'),
    row=1, col=1
)

# Priority Score Box
for label in CATEGORIES:
    fig_plotly.add_trace(
        go.Box(
            y=df[df['label'] == label]['priority_score'],
            name=label, marker_color=COLORS[label], showlegend=False
        ),
        row=1, col=2
    )

fig_plotly.update_layout(
    title_text='🤖 Smart Email Management System — Interactive Dashboard',
    template='plotly_dark',
    height=500,
    barmode='group',
    font=dict(size=11)
)
fig_plotly.update_yaxes(range=[0, 120], row=1, col=1)
fig_plotly.show()

# ── 2. Interactive Sunburst Chart ──
priority_map = {
    '🔴 CRITICAL': 'CRITICAL', '🟠 HIGH': 'HIGH',
    '🟡 MEDIUM': 'MEDIUM', '🟢 LOW': 'LOW', '⚪ NEGLIGIBLE': 'NEGLIGIBLE'
}
df['priority_clean'] = df['priority_label'].map(priority_map)

sunburst_data = df.groupby(['label', 'priority_clean']).size().reset_index(name='count')

fig_sun = px.sunburst(
    sunburst_data,
    path=['label', 'priority_clean'],
    values='count',
    title='📊 Email Category → Priority Sunburst Chart',
    color='label',
    color_discrete_map=COLORS,
    template='plotly_dark'
)
fig_sun.update_layout(height=500)
fig_sun.show()

print("✅ Interactive Charts Generated!")

---
## 💬 SECTION 11: Interactive Email Analysis Interface
---

In [ ]:
# ============================================================
# CELL 15: Full Interactive Email Analysis System
# ============================================================

print("💬 Building Interactive Email Analysis Interface...\n")

class SmartEmailAnalyzer:
    """
    Complete Smart Email Management System.
    Integrates: Classification + Priority Scoring + Summarization.
    """

    def __init__(self, classifier_model, vectorizer, preprocessor,
                 label_encoder, priority_scorer, summarizer_fn):
        self.classifier     = classifier_model
        self.vectorizer     = vectorizer
        self.preprocessor   = preprocessor
        self.label_encoder  = label_encoder
        self.priority_scorer = priority_scorer
        self.summarize       = summarizer_fn

    def analyze(self, email_text, subject='', sender='', use_bert=False):
        """
        Full analysis pipeline for a single email.
        Returns structured results dict.
        """
        # ── 1. Preprocess ──
        full_text   = f"{subject} {email_text}".strip() if subject else email_text
        processed   = self.preprocessor.preprocess(full_text)

        # ── 2. Classify with ML ──
        tfidf_vec   = self.vectorizer.transform([processed])
        pred_idx    = self.classifier.predict(tfidf_vec)[0]
        pred_label  = self.label_encoder.inverse_transform([pred_idx])[0]

        # Confidence scores
        if hasattr(self.classifier, 'predict_proba'):
            proba = self.classifier.predict_proba(tfidf_vec)[0]
        else:
            # For LinearSVC, use decision function
            scores = self.classifier.decision_function(tfidf_vec)[0]
            proba  = np.exp(scores) / np.exp(scores).sum()

        confidence = float(proba.max() * 100)
        all_probs  = dict(zip(self.label_encoder.classes_, (proba * 100).round(2)))

        # ── 3. BERT Prediction (optional) ──
        bert_label = None
        if use_bert:
            bert_model.eval()
            enc = bert_tokenizer(
                full_text[:512], max_length=128,
                padding='max_length', truncation=True, return_tensors='pt'
            )
            with torch.no_grad():
                out = bert_model(
                    enc['input_ids'].to(device),
                    enc['attention_mask'].to(device)
                )
                bert_pred_idx = out.logits.argmax(dim=1).item()
                bert_label    = self.label_encoder.inverse_transform([bert_pred_idx])[0]

        # ── 4. Priority Score ──
        priority_result = self.priority_scorer.compute_priority_score(full_text, pred_label)
        priority_score  = priority_result['priority_score']
        priority_level  = self.priority_scorer.get_priority_label(priority_score)

        # ── 5. Summarize ──
        summary = self.summarize(full_text)

        return {
            'category':         pred_label,
            'confidence':       confidence,
            'all_probabilities': all_probs,
            'bert_prediction':  bert_label,
            'priority_score':   priority_score,
            'priority_level':   priority_level,
            'matched_keywords': priority_result['matched_keywords'],
            'summary':          summary,
            'word_count':       len(full_text.split()),
        }

    def display_result(self, result, email_text, subject='', sender=''):
        """Pretty-print the analysis result."""
        sep = "═" * 65
        print(f"\n{sep}")
        print("  🤖 SMART EMAIL ANALYSIS REPORT")
        print(sep)

        if sender:  print(f"  👤 From     : {sender}")
        if subject: print(f"  📌 Subject  : {subject}")
        print(f"  📝 Preview  : {email_text[:80]}..." if len(email_text) > 80 else f"  📝 Email: {email_text}")
        print(f"  📏 Words    : {result['word_count']}")
        print()

        # Category
        category_icons = {
            'Important': '🔴', 'Spam': '⚠️', 'Promotional': '🛍️',
            'Social': '👥', 'Updates': '🔔'
        }
        icon = category_icons.get(result['category'], '📧')
        print(f"  {icon} Category    : {result['category'].upper()}")
        print(f"  📊 Confidence   : {result['confidence']:.1f}%")
        if result['bert_prediction']:
            print(f"  🧠 BERT Pred    : {result['bert_prediction']}")

        print()
        print("  📈 Category Probabilities:")
        for cat, prob in sorted(result['all_probabilities'].items(),
                                 key=lambda x: x[1], reverse=True):
            bar = '█' * int(prob / 5)
            print(f"     {cat:<15}: {prob:>6.2f}% {bar}")

        print()
        print(f"  🎯 Priority Score : {result['priority_score']}/100")
        print(f"  🚦 Priority Level : {result['priority_level']}")
        if result['matched_keywords']:
            print(f"  🔑 Keywords Found : {', '.join(result['matched_keywords'])}")

        print()
        print(f"  📝 AI Summary:")
        print(f"     {result['summary']}")
        print(sep)


# ── Initialize Analyzer ──
analyzer = SmartEmailAnalyzer(
    classifier_model = trained_models[best_model_name],
    vectorizer       = tfidf_vectorizer,
    preprocessor     = preprocessor,
    label_encoder    = label_encoder,
    priority_scorer  = priority_scorer,
    summarizer_fn    = generate_summary
)

print("✅ Smart Email Analyzer initialized!")
print(f"   Using: {best_model_name} as primary classifier")
print(f"   Accuracy: {best_acc*100:.2f}%")

In [ ]:
# ============================================================
# CELL 16: Analyze Diverse Test Email Samples
# ============================================================

print("📧 Running Analysis on Test Email Samples...\n")

test_emails = [
    {
        'subject': 'URGENT: Server Down - Production Impact',
        'sender':  'operations@company.com',
        'text': """Hi Team, The production server has been down for the past 30 minutes and we are experiencing a complete outage. This is a P0 incident affecting over 50,000 active users. All engineers must join the incident call immediately. The CTO and client representatives are already on the bridge. We need the hotfix deployed within the next 2 hours or we will breach our SLA. Please respond ASAP to confirm your availability."""
    },
    {
        'subject': 'You Won $1,000,000!!! Claim NOW!!!',
        'sender':  'noreply@lucky-winners.net',
        'text': """CONGRATULATIONS DEAR WINNER!!! You have been selected as the GRAND PRIZE WINNER of our international lottery! Your email address won you ONE MILLION DOLLARS! To claim your prize immediately, send us your bank account details, social security number, and a processing fee of $500. This offer expires in 24 HOURS! Don't miss your chance to become a millionaire overnight! Reply to this email or click the link to claim YOUR MONEY NOW!!!"""
    },
    {
        'subject': 'MEGA SALE: 70% off everything this weekend!',
        'sender':  'deals@shopworld.com',
        'text': """Hey valued customer! Our biggest sale of the year is happening this weekend only! Get up to 70% off on electronics, fashion, home appliances, and more. Use code MEGA70 for an additional discount at checkout. Free delivery on all orders above $50. Limited stock available — shop now before items sell out! Browse 10,000+ products from top brands. Sale ends Sunday at midnight!"""
    },
    {
        'subject': 'Happy Birthday! Your friends are celebrating you!',
        'sender':  'notifications@socialconnect.com',
        'text': """Hey! Today is your special day and your friends are thinking of you! Priya, Amit, Sneha and 23 others have posted birthday wishes on your profile. You have 28 new notifications, 15 messages, and 4 gift requests waiting. Log in now to see all your birthday greetings and celebrate with your social circle. Your friend Rahul also sent you a virtual gift!"""
    },
    {
        'subject': 'Your order #ORD847291 has been shipped',
        'sender':  'shipping@amazon.com',
        'text': """Hello! Great news — your order #ORD847291 has been shipped and is on its way to you. Your package containing MacBook Pro 14-inch and accessories will be delivered by tomorrow between 9 AM and 6 PM. Track your shipment using the TrackNow app or click the button below. If you have any delivery issues, contact our 24/7 support team."""
    },
    {
        'subject': 'Interview Confirmation - Software Engineer Role at Google',
        'sender':  'recruiting@google.com',
        'text': """Dear Candidate, We are pleased to confirm your technical interview for the Software Engineer position at Google. Your interview is scheduled for this Friday at 2:00 PM IST via Google Meet. The interview panel consists of 3 senior engineers and will cover system design, algorithms, and coding problems. Please ensure you have a stable internet connection and a quiet environment. Bring digital copies of your resume. Respond to confirm your availability by tomorrow evening."""
    },
]

for email in test_emails:
    result = analyzer.analyze(
        email_text=email['text'],
        subject=email['subject'],
        sender=email['sender'],
        use_bert=False  # Set True to also run BERT prediction
    )
    analyzer.display_result(result, email['text'], email['subject'], email['sender'])
    print()

In [ ]:
# ============================================================
# CELL 17: Interactive User Input — Enter Your Own Email
# ============================================================

print("\n" + "="*65)
print("  💬 INTERACTIVE EMAIL ANALYSIS")
print("  Enter your own email below to analyze it!")
print("="*65)

# ── Default email (change this to test your own!) ──
USER_SUBJECT = "Meeting Tomorrow - Project Status Update Required"
USER_SENDER  = "manager@company.com"
USER_EMAIL   = """
Hi Team,

I hope this email finds you well. I wanted to remind everyone that we have
an important project status meeting scheduled for tomorrow morning at 10 AM.
This is mandatory for all team members.

Please come prepared with your weekly progress report, any blockers you are
facing, and an updated timeline for your deliverables. The project deadline
is approaching on Friday and we need to ensure everything is on track.

The CEO will also be joining us for the last 15 minutes, so please be
professional and have your key metrics ready.

Please confirm your attendance by tonight.

Best regards,
Rajesh Kumar
Project Manager
"""

# ── Analyze ──
user_result = analyzer.analyze(
    email_text=USER_EMAIL,
    subject=USER_SUBJECT,
    sender=USER_SENDER,
    use_bert=False
)
analyzer.display_result(user_result, USER_EMAIL, USER_SUBJECT, USER_SENDER)

print("\n💡 TIP: Modify USER_EMAIL, USER_SUBJECT, USER_SENDER above to test your own emails!")

---
## 🛡️ SECTION 12: Advanced Features — Batch Processing & Stats
---

In [ ]:
# ============================================================
# CELL 18: Batch Email Processing & Inbox Analytics
# ============================================================

print("📬 Simulating Batch Inbox Processing...\n")

# ── Simulate Inbox ──
inbox_emails = [
    {"id": "E001", "from": "boss@company.com",      "subject": "Quarterly Report Due Tomorrow",       "text": "Please submit the quarterly financial report by tomorrow 5 PM. The board meeting is on Friday and this is critical for the presentation."},
    {"id": "E002", "from": "noreply@bank.com",        "subject": "OTP for transaction",                "text": "Your OTP for the transaction of $2,500 is 847291. Valid for 10 minutes. Do not share with anyone."},
    {"id": "E003", "from": "deals@flipkart.com",      "subject": "Big Billion Days Sale!",              "text": "The biggest sale of the year is here! Up to 80% off on mobiles, laptops, TVs and more. Flash deals every hour. Shop now and save big!"},
    {"id": "E004", "from": "friend@gmail.com",        "subject": "Weekend plans?",                      "text": "Hey! A bunch of us are planning to go hiking this Saturday. Want to join? Let me know by Thursday so I can book the cabs."},
    {"id": "E005", "from": "hr@company.com",          "subject": "Salary Slip - November 2024",         "text": "Your salary for November 2024 has been processed. Please find the attached payslip. Net amount: ₹85,000 credited to your account."},
    {"id": "E006", "from": "scam@fakeprize.com",      "subject": "You won FREE iPhone!",                "text": "CONGRATULATIONS! You are today's lucky winner of a FREE iPhone 15 Pro! Click here to claim immediately. Limited offer. Only 5 left!"},
    {"id": "E007", "from": "linkedin@linkedin.com",   "subject": "10 new job recommendations",          "text": "Based on your profile, we found 10 new jobs matching your skills in Python and Machine Learning. Companies include Google, Amazon, Microsoft."},
    {"id": "E008", "from": "alerts@github.com",       "subject": "Security alert: new sign-in",          "text": "A new sign-in to your GitHub account was detected from Bangalore, India. If this was you, no action needed. Otherwise, secure your account now."},
    {"id": "E009", "from": "professor@college.edu",   "subject": "URGENT: Viva schedule changed",        "text": "Dear students, your final year project viva has been rescheduled to this Friday at 10 AM. Attendance is compulsory. Come with your presentation ready."},
    {"id": "E010", "from": "amazon@email.amazon.com", "subject": "Your order has been delivered",        "text": "Great news! Your order has been successfully delivered. Order #112-8472910. If you have any issues with the product, contact us within 30 days."},
]

print(f"{'ID':<6} {'FROM':<30} {'CATEGORY':<14} {'CONFIDENCE':<12} {'PRIORITY':<8} {'LEVEL'}")
print("-"*95)

batch_results = []
for email in inbox_emails:
    res = analyzer.analyze(
        email_text=email['text'],
        subject=email['subject'],
        sender=email['from']
    )
    category_icons = {'Important':'🔴','Spam':'⚠️ ','Promotional':'🛍️','Social':'👥 ','Updates':'🔔 '}
    icon = category_icons.get(res['category'], '📧')
    print(
        f"{email['id']:<6} "
        f"{email['from'][:28]:<30} "
        f"{icon}{res['category']:<12} "
        f"{res['confidence']:>8.1f}%    "
        f"{res['priority_score']:>5}/100  "
        f"{res['priority_level']}"
    )
    batch_results.append({**email, **res})

print("-"*95)
batch_df = pd.DataFrame(batch_results)

print("\n📊 Inbox Summary:")
print(f"   Total Emails     : {len(batch_df)}")
print(f"   Spam Detected    : {(batch_df['category']=='Spam').sum()}")
print(f"   Important Emails : {(batch_df['category']=='Important').sum()}")
print(f"   Avg Priority     : {batch_df['priority_score'].mean():.1f}/100")
print(f"   Critical Emails  : {(batch_df['priority_score'] >= 80).sum()}")
print("\n✅ Batch Processing Complete!")

---
## 🚀 SECTION 13: Error Analysis & Model Insights
---

In [ ]:
# ============================================================
# CELL 19: Error Analysis & Misclassification Insights
# ============================================================

print("🔍 Error Analysis & Misclassification Study...\n")

# Get predictions on full test set
best_preds_all = trained_models[best_model_name].predict(X_test_tfidf)
y_test_labels  = label_encoder.inverse_transform(y_test)
pred_labels    = label_encoder.inverse_transform(best_preds_all)

# Build error analysis dataframe
test_df = pd.DataFrame({
    'text':       X_test_raw,
    'true_label': y_test_labels,
    'pred_label': pred_labels,
    'correct':    y_test_labels == pred_labels
})

errors_df = test_df[~test_df['correct']].copy()

print(f"📊 Error Analysis for {best_model_name}:")
print(f"   Total Test Samples : {len(test_df)}")
print(f"   Correctly Classified : {test_df['correct'].sum()} ({test_df['correct'].mean()*100:.1f}%)")
print(f"   Misclassified : {(~test_df['correct']).sum()} ({(~test_df['correct']).mean()*100:.1f}%)")

print("\n📋 Most Common Misclassification Patterns:")
print("-"*65)
misclass = errors_df.groupby(['true_label','pred_label']).size().sort_values(ascending=False)
for (true_l, pred_l), count in misclass.head(10).items():
    print(f"   {true_l:<15} → {pred_l:<15} : {count} cases")

print("\n📋 Per-Class Error Analysis:")
print("-"*65)
for label in CATEGORIES:
    label_df     = test_df[test_df['true_label'] == label]
    label_errors = label_df[~label_df['correct']]
    if len(label_df) > 0:
        error_rate = len(label_errors)/len(label_df)*100
        bar = '█' * int(error_rate / 5)
        print(f"   {label:<15}: {len(label_errors):>3}/{len(label_df):>3} errors ({error_rate:.1f}%) {bar}")

# Show misclassified examples
print("\n📧 Sample Misclassified Emails:")
print("-"*65)
if len(errors_df) > 0:
    for _, row in errors_df.head(3).iterrows():
        print(f"  True: {row['true_label']} | Predicted: {row['pred_label']}")
        print(f"  Text: {row['text'][:120]}...")
        print()

print("💡 Analysis Insights:")
print("  • Promotional and Spam emails share urgency language → confusion")
print("  • Important and Updates both use formal language → overlap")
print("  • Social emails with professional content may be tagged as Important")
print("  • BERT contextual embeddings reduce these overlaps significantly")

In [ ]:
# ============================================================
# CELL 20: Feature Importance Analysis
# ============================================================

print("📊 Feature Importance Analysis — Top Discriminating Terms\n")

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.patch.set_facecolor('#0A0E1A')
axes = axes.flatten()

# Use Logistic Regression coefficients for interpretability
lr_model = trained_models['Logistic Regression']
feature_names = tfidf_vectorizer.get_feature_names_out()

for idx, (label, color) in enumerate(COLORS.items()):
    ax = axes[idx]
    ax.set_facecolor('#0F1620')

    class_idx = list(CATEGORIES).index(label)
    coef = lr_model.coef_[class_idx]

    # Top positive and negative terms
    top_pos_idx = coef.argsort()[-15:][::-1]
    top_neg_idx = coef.argsort()[:5]

    top_pos_terms  = [feature_names[i] for i in top_pos_idx]
    top_pos_scores = [coef[i] for i in top_pos_idx]
    top_neg_terms  = [feature_names[i] for i in top_neg_idx]
    top_neg_scores = [coef[i] for i in top_neg_idx]

    all_terms  = top_neg_terms + top_pos_terms
    all_scores = top_neg_scores + top_pos_scores

    bar_colors = ['#E63946' if s < 0 else color for s in all_scores]
    y_pos = range(len(all_terms))

    ax.barh(y_pos, all_scores, color=bar_colors, edgecolor='none', alpha=0.85)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(all_terms, fontsize=8, color='#CCCCCC')
    ax.axvline(x=0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_title(f'{label} — Top Discriminating Terms',
                 color='white', fontsize=11, fontweight='bold')
    ax.set_xlabel('LR Coefficient', color='#CCCCCC', fontsize=9)
    ax.tick_params(colors='#CCCCCC')
    ax.grid(color='#1E2A3A', linestyle='--', linewidth=0.5, alpha=0.7)
    for spine in ax.spines.values():
        spine.set_edgecolor('#1E2A3A')

# Hide last subplot if odd number of categories
if len(CATEGORIES) < len(axes):
    axes[-1].set_visible(False)

fig.suptitle('🔠 Feature Importance — Top Discriminating Terms per Category\n(Logistic Regression Coefficients)',
             color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0A0E1A')
plt.show()
print("✅ Feature Importance Analysis Complete!")